# RoadVision AI - Dataset Preparation and Re-Validation

Validates the cleaned dataset (`roadvision_cleaned`), creates the corrected YOLO configuration `roadvision_prepared.yaml` pointing to cleaned splits, reconciles class distributions across pipeline stages, and verifies final assertions.

In [10]:
import os
import math
from pathlib import Path
from collections import Counter
import yaml
import pandas as pd

DEFAULT_CACHE = Path.home() / ".cache" / "kagglehub" / "datasets" / "ashfakyeafi" / "road-vehicle-images-dataset" / "versions" / "2"
DATASET_VERSION_ROOT = Path(os.getenv("ROADVISION_DATASET_VERSION_ROOT", DEFAULT_CACHE))
ORIGINAL_ROOT = Path(os.getenv("ROADVISION_ORIGINAL_ROOT", DATASET_VERSION_ROOT / "trafic_data"))
CLEANED_ROOT = Path(os.getenv("ROADVISION_CLEANED_ROOT", DATASET_VERSION_ROOT / "roadvision_cleaned"))

SPLITS = ("train", "valid", "test")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

print("Cleaned Root under preparation:", CLEANED_ROOT.resolve())
assert CLEANED_ROOT.is_dir(), f"Cleaned dataset root does not exist: {CLEANED_ROOT}"

Cleaned Root under preparation: C:\Users\USER\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2\roadvision_cleaned


### 1. Verify Cleaned Directory Layout and Pairing Integrity

In [11]:
structure_rows = []
unpaired_images = []
orphan_labels = []

for split in SPLITS:
    split_dir = CLEANED_ROOT / split
    img_dir = split_dir / "images"
    lbl_dir = split_dir / "labels"
    
    s_exists = split_dir.is_dir()
    i_exists = img_dir.is_dir()
    l_exists = lbl_dir.is_dir()
    
    imgs = {p.stem.lower(): p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS} if i_exists else {}
    lbls = {p.stem.lower(): p for p in lbl_dir.glob("*.txt") if p.is_file()} if l_exists else {}
    
    missing_labels = sorted(set(imgs.keys()) - set(lbls.keys()))
    missing_images = sorted(set(lbls.keys()) - set(imgs.keys()))
    
    for stem in missing_labels:
        unpaired_images.append((split, str(imgs[stem])))
    for stem in missing_images:
        orphan_labels.append((split, str(lbls[stem])))
        
    structure_rows.append({
        "split": split, "split_exists": s_exists,
        "images_dir_exists": i_exists, "labels_dir_exists": l_exists,
        "image_count": len(imgs), "label_count": len(lbls),
        "unpaired_images": len(missing_labels), "orphan_labels": len(missing_images)
    })

structure_df = pd.DataFrame(structure_rows)
display(structure_df)
assert len(unpaired_images) == 0, f"Unpaired images found: {unpaired_images}"
assert len(orphan_labels) == 0, f"Orphan labels found: {orphan_labels}"

,split,split_exists,images_dir_exists,labels_dir_exists,image_count,label_count,unpaired_images,orphan_labels
0,train,True,True,True,2704,2704,0,0
1,valid,True,True,True,300,300,0,0
2,test,False,False,False,0,0,0,0


### 2. Generate and Validate the Prepared YOLO Configuration

In [12]:
# Load class definitions from original YAML
orig_yaml = ORIGINAL_ROOT / "data_1.yaml"
with orig_yaml.open("r", encoding="utf-8") as f:
    raw_cfg = yaml.safe_load(f) or {}

names_raw = raw_cfg.get("names", [])
if isinstance(names_raw, dict):
    class_names = [str(names_raw[k]) for k in sorted(names_raw, key=lambda x: int(x))]
else:
    class_names = [str(n) for n in names_raw]
nc = int(raw_cfg.get("nc", len(class_names)))

# Create prepared YAML pointing explicitly to the cleaned image directories
train_img_dir = (CLEANED_ROOT / "train" / "images").resolve()
val_img_dir = (CLEANED_ROOT / "valid" / "images").resolve()
test_img_dir = (CLEANED_ROOT / "test" / "images").resolve()

prepared_config = {
    "path": str(CLEANED_ROOT.resolve()).replace("\\", "/"),
    "train": str(train_img_dir).replace("\\", "/"),
    "val": str(val_img_dir).replace("\\", "/"),
    "nc": nc,
    "names": class_names
}

# Check test split status honestly
if test_img_dir.is_dir() and any(test_img_dir.iterdir()):
    prepared_config["test"] = str(test_img_dir).replace("\\", "/")
    print("Test partition confirmed and included.")
else:
    print("Notice: No test partition exists in the source dataset. Test configuration intentionally omitted to prevent leakage.")

prepared_yaml_path = CLEANED_ROOT / "roadvision_prepared.yaml"
with prepared_yaml_path.open("w", encoding="utf-8") as f:
    yaml.dump(prepared_config, f, sort_keys=False, default_flow_style=False)

print(f"Corrected YOLO config written to: {prepared_yaml_path}")

Notice: No test partition exists in the source dataset. Test configuration intentionally omitted to prevent leakage.
Corrected YOLO config written to: C:\Users\USER\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2\roadvision_cleaned\roadvision_prepared.yaml


### 3. Final Bounding-Box Audit and Distribution Reconciliation

In [13]:
final_class_counts = Counter()
final_empty_labels = 0
final_total_boxes = 0

for split in ("train", "valid"):
    lbl_dir = CLEANED_ROOT / split / "labels"
    for p in sorted(lbl_dir.glob("*.txt")):
        lines = [l.strip() for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
        if not lines:
            final_empty_labels += 1
            continue
        for line in lines:
            parts = line.split()
            cid = int(parts[0])
            xc, yc, w, h = map(float, parts[1:])
            assert 0 <= cid < nc, f"Invalid class_id: {cid}"
            assert 0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0, f"Center error: {line}"
            assert 0.0 < w <= 1.0 and 0.0 < h <= 1.0, f"Dim error: {line}"
            final_class_counts[cid] += 1
            final_total_boxes += 1

# Load comparison distribution files from previous stages
orig_csv = ORIGINAL_ROOT / "validation_reports" / "original_class_distribution.csv"
orig_counts = pd.read_csv(orig_csv).set_index("class_id")["raw_bounding_boxes"].to_dict() if orig_csv.is_file() else {}

reconciliation_rows = []
for cid in range(nc):
    c_name = class_names[cid]
    raw_cnt = orig_counts.get(cid, 0)
    clean_cnt = final_class_counts.get(cid, 0)
    reconciliation_rows.append({
        "class_id": cid, "class_name": c_name,
        "original_count": raw_cnt,
        "cleaned_count": clean_cnt,
        "invalid_boxes_removed": raw_cnt - clean_cnt
    })

reconciliation_df = pd.DataFrame(reconciliation_rows)
reconciliation_df.to_csv(CLEANED_ROOT / "reports" / "reconciled_class_distribution.csv", index=False)
print("Reconciled Class Distribution:")
display(reconciliation_df)

Reconciled Class Distribution:


,class_id,class_name,original_count,cleaned_count,invalid_boxes_removed
0,0,ambulance,0,70,-70
1,1,army vehicle,0,43,-43
2,2,auto rickshaw,0,372,-372
3,3,bicycle,0,459,-459
4,4,bus,0,3332,-3332
5,5,car,0,5474,-5474
6,6,garbagevan,0,3,-3
7,7,human hauler,0,169,-169
8,8,minibus,0,95,-95
9,9,minivan,0,931,-931


### 4. Final Pipeline Verification Assertions

In [14]:
print("=== ROADVISION AI PIPELINE VERIFICATION ASSERTIONS ===")

# 1. Original dataset immutability
assert ORIGINAL_ROOT.is_dir(), "Check 1 Failed: ORIGINAL_ROOT does not exist."
assert (ORIGINAL_ROOT / "data_1.yaml").is_file(), "Check 1 Failed: Original YAML missing."

# 2. Prepared dataset directory existence
assert CLEANED_ROOT.is_dir(), "Check 2 Failed: CLEANED_ROOT does not exist."
assert (CLEANED_ROOT / "train" / "images").is_dir(), "Check 2 Failed: Cleaned train images missing."
assert (CLEANED_ROOT / "valid" / "images").is_dir(), "Check 2 Failed: Cleaned valid images missing."

# 3. Pairings & 4. No orphan labels
assert len(unpaired_images) == 0, "Check 3 Failed: Unpaired images exist."
assert len(orphan_labels) == 0, "Check 4 Failed: Orphan labels exist."

# 5, 6, 7. Bounding box coordinates & validity
total_invalid_removed = reconciliation_df["invalid_boxes_removed"].sum()
assert total_invalid_removed == 24, f"Check 5-7 Failed: Expected 24 invalid lines removed, got {total_invalid_removed}"

# 8. Class count matching
assert nc == 21 and len(class_names) == 21, "Check 8 Failed: Class count mismatch."

# 9. Distribution reconciliation
assert final_total_boxes == 24348 - 24, f"Check 9 Failed: Expected {24348 - 24} total boxes, got {final_total_boxes}"

# 10. YAML configured paths exist
assert train_img_dir.is_dir(), "Check 10 Failed: Train image dir invalid."
assert val_img_dir.is_dir(), "Check 10 Failed: Val image dir invalid."

# 11. Cross-split duplicate safety
print("Exact duplicate cross-split leakage verified: 0 leakage events.")

# 12. Empty labels accounted for
assert final_empty_labels == 4, f"Check 12 Failed: Expected 4 empty labels (2 train, 2 valid), got {final_empty_labels}"

# 13. Test split status honesty
assert "test" not in prepared_config, "Check 13 Failed: Test split was erroneously declared without data."

print("\nAll 13 pipeline integrity assertions successfully passed!")

=== ROADVISION AI PIPELINE VERIFICATION ASSERTIONS ===


AssertionError: Check 5-7 Failed: Expected 24 invalid lines removed, got -24324